In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision.transforms as transforms
import torchvision.models as models
import torch.utils.data as data
import glob
import math
import pandas as pd
import importlib
import cv2
from pathlib import Path
from collections import OrderedDict
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm
import matplotlib.pyplot as plt
import timm

# ==============================================================================
# SECTION 1: DATA LOADING
# ==============================================================================

def np_load_frame(filename, resize_height, resize_width):
    """Loads and preprocesses a single image frame from a file path."""
    image_decoded = cv2.imread(filename)
    if image_decoded is None:
        print(f"Error: Could not read image {filename}")
        return None, 0, 0
    image_resized = cv2.resize(image_decoded, (resize_width, resize_height))
    image_resized = image_resized.astype(dtype=np.float32)
    image_resized = (image_resized / 127.5) - 1.0
    return image_resized, 0, 0

class DataLoader(data.Dataset):
    """Custom DataLoader to load sequences of video frames."""
    def __init__(self, video_folder, transform, resize_height, resize_width, time_step=4, num_pred=1):
        self.dir = video_folder
        self.transform = transform
        self.video_frames = []
        self._resize_height = resize_height
        self._resize_width = resize_width
        self._time_step = time_step
        self._num_pred = num_pred
        self.index_samples = []
        self.setup()

    def setup(self):
        """Finds all video frames and creates sample indices."""
        videos = glob.glob(os.path.join(self.dir, '*'))
        if not videos:
            print(f"Warning: No files or directories found in {self.dir}")
            return
        
        videos.sort()
        all_video_frames = []
        if os.path.isdir(videos[0]):
            for video in videos:
                vide_frames = glob.glob(os.path.join(video, '*.jpg'))
                vide_frames.sort(key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))
                all_video_frames.extend(vide_frames)
        else:
            image_files = [f for f in videos if f.lower().endswith(('.png', '.jpg', 'jpeg'))]
            image_files.sort(key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))
            all_video_frames = image_files
        
        self.video_frames = all_video_frames
        self.index_samples = list(range(len(all_video_frames) - (self._time_step + self._num_pred) + 1))

    def __getitem__(self, index):
        frame_index_start = self.index_samples[index]
        batch_frames_standard = np.zeros((self._time_step + self._num_pred, 3, self._resize_height, self._resize_width), dtype=np.float32)
        for i in range(self._time_step + self._num_pred):
            frame_path = self.video_frames[frame_index_start + i]
            image_standard, _, _ = np_load_frame(frame_path, self._resize_height, self._resize_width)
            if self.transform is not None:
                batch_frames_standard[i] = self.transform(image_standard)
        return {'standard': torch.from_numpy(batch_frames_standard)}

    def __len__(self):
        return len(self.index_samples)

# ==============================================================================
# SECTION 2: CONFIGURATION LOGIC
# ==============================================================================

def print_config(config):
    """Prints the configuration parameters in a formatted way."""
    message = '----------------- Config ---------------\n'
    for k, v in sorted(vars(config).items()):
        message += f'{str(k):>35}: {str(v):<30}\n'
    message += '----------------- End -------------------'
    print(message)

def get_train_config():
    """Returns the main configuration object for training."""
    class Config:
        exp_name = "SOTA_VAD_with_PerceptualLoss"
        train = 1
        n_gpu = 1
        tensorboard = False
        image_size = 224
        batch_size = 8
        num_frames = 4
        num_workers = 1
        epochs = 10
        lr = 1e-4
        wd = 1e-4
        warmup_steps = 100
        
        # Explicit weights for a clearer, more flexible loss combination
        l1_weight = 0.10
        ssim_weight = 0.85
        perceptual_weight = 0.05

        summary_dir = f"experiments/tb/{exp_name}"
    config = Config()
    print_config(config)
    return config

# ==============================================================================
# SECTION 3: UTILITIES
# ==============================================================================

def setup_device(n_gpu_use):
    """Sets up the device (GPU/CPU) for training."""
    n_gpu = torch.cuda.device_count()
    if n_gpu_use > 0 and n_gpu == 0:
        print("Warning: There's no GPU available, training will be on CPU.")
        n_gpu_use = 0
    device = torch.device('cuda:0' if n_gpu_use > 0 else 'cpu')
    return device, list(range(n_gpu_use))

class MetricTracker:
    def __init__(self, *keys):
        self._data = pd.DataFrame(index=keys, columns=['total', 'counts', 'average'])
        self.reset()
    def reset(self):
        for col in self._data.columns:
            self._data[col].values[:] = 0
    def update(self, key, value, n=1):
        self._data.total[key] += value * n
        self._data.counts[key] += n
        self._data.average[key] = self._data.total[key] / self._data.counts[key]
    def result(self):
        return dict(self._data.average)

# ==============================================================================
# SECTION 4: MODEL ARCHITECTURE & LOSSES
# ==============================================================================

def gaussian_window(size, sigma):
    """Creates a 1D Gaussian window."""
    coords = torch.arange(size, dtype=torch.float)
    coords -= size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    return g / g.sum()

def create_window(window_size, channel):
    """Creates a 2D Gaussian window."""
    _1D_window = gaussian_window(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

class SSIM(nn.Module):
    """A self-contained, corrected SSIM class."""
    def __init__(self, window_size=11, size_average=True):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = create_window(window_size, self.channel)

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()
        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = create_window(self.window_size, channel)
            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)
            self.window = window
            self.channel = channel

        padding = self.window_size // 2
        mu1 = F.conv2d(img1, window, padding=padding, groups=channel)
        mu2 = F.conv2d(img2, window, padding=padding, groups=channel)
        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2
        sigma1_sq = F.conv2d(img1 * img1, window, padding=padding, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2 * img2, window, padding=padding, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1 * img2, window, padding=padding, groups=channel) - mu1_mu2
        C1 = 0.01 ** 2
        C2 = 0.03 ** 2
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

        if self.size_average:
            return ssim_map.mean()
        else:
            return ssim_map.mean(1).mean(1).mean(1)

class PerceptualLoss(nn.Module):
    """VGG-based perceptual loss."""
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg19(pretrained=True).features
        self.slices = nn.ModuleList([vgg[i] for i in range(36)]) # Up to conv4_4
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, input_img, target_img):
        # Normalize for VGG
        mean = torch.tensor([0.485, 0.456, 0.406], device=input_img.device).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=input_img.device).view(1, 3, 1, 1)
        input_norm = (input_img - mean) / std
        target_norm = (target_img - mean) / std
        
        input_features = self.get_features(input_norm)
        target_features = self.get_features(target_norm)

        loss = 0.0
        for f_input, f_target in zip(input_features, target_features):
            loss += F.l1_loss(f_input, f_target)
        return loss
    
    def get_features(self, x):
        features = []
        # Layers for feature extraction from VGG19
        feature_layers = {3, 8, 17, 26, 35} 
        for i, layer in enumerate(self.slices):
            x = layer(x)
            if i in feature_layers:
                features.append(x)
        return features

class VisionTransformerSOTA(nn.Module):
    """
    The advanced model using a pre-trained ViT encoder.
    This version is CORRECTED to use the spatial patch embeddings from the ViT,
    not just the single CLS token, leading to a much richer input for the decoder.
    The decoder has also been enhanced for better reconstruction quality.
    """
    def __init__(self, image_size=224, num_frames=4):
        super(VisionTransformerSOTA, self).__init__()
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=True)
        emb_dim = self.encoder.head.in_features
        
        for param in list(self.encoder.parameters())[:-40]:
            param.requires_grad = False

        feature_map_size = image_size // 16 
        decoder_input_channels = emb_dim * num_frames

        self.decoder_net = nn.Sequential(
            nn.Conv2d(decoder_input_channels, 512, kernel_size=1),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.ConvTranspose2d(64, 3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        batch_size, num_frames, C, H, W = x.shape
        frame_features_list = []
        feature_map_size = H // 16

        for i in range(num_frames):
            features = self.encoder.forward_features(x[:, i, :, :, :])
            patch_tokens = features[:, 1:, :] 
            patch_features = patch_tokens.permute(0, 2, 1).reshape(batch_size, -1, feature_map_size, feature_map_size)
            frame_features_list.append(patch_features)
            
        combined_features = torch.cat(frame_features_list, dim=1)
        reconstructed_image = self.decoder_net(combined_features)
        
        return reconstructed_image

# ==============================================================================
# SECTION 5: VISUALIZATION & PLOTTING
# ==============================================================================
def save_anomaly_visualization(gt_frame, pred_frame, epoch, batch_idx, loss, save_dir):
    gt_img = (gt_frame.squeeze(0).permute(1, 2, 0).contiguous().cpu().numpy() + 1.0) * 127.5
    pred_img = (pred_frame.squeeze(0).permute(1, 2, 0).contiguous().cpu().numpy() + 1.0) * 127.5
    gt_img = gt_img.astype(np.uint8)
    pred_img = pred_img.astype(np.uint8)
    residual = cv2.absdiff(gt_img, pred_img)
    residual_gray = cv2.cvtColor(residual, cv2.COLOR_BGR2GRAY)
    residual_color = cv2.applyColorMap(residual_gray, cv2.COLORMAP_JET)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(gt_img, 'Ground Truth', (10, 25), font, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(pred_img, 'Prediction', (10, 25), font, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(residual_color, 'Residual', (10, 25), font, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
    combined_img = np.concatenate((gt_img, pred_img, residual_color), axis=1)
    file_path = os.path.join(save_dir, f"epoch_{epoch}_frame_{batch_idx}.jpg")
    cv2.imwrite(file_path, combined_img)

def save_single_roc_curve(epoch_data, filename, title=None):
    """Saves a single ROC curve plot. Can accept an optional custom title."""
    plt.figure(figsize=(10, 8))
    
    if title is None:
        plot_title = f"ROC Curve for Epoch {epoch_data['epoch']}"
    else:
        plot_title = title

    plt.plot(epoch_data['fpr'], epoch_data['tpr'], lw=2, color='darkorange',
             label=f"Epoch {epoch_data['epoch']} (AUC = {epoch_data['auc']:.4f})")
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(plot_title)
    plt.legend(loc="lower right"); plt.grid(True)
    plt.savefig(filename); plt.close()

def save_combined_roc_curves(history, filename):
    plt.figure(figsize=(12, 9))
    colors = plt.cm.viridis(np.linspace(0, 1, len(history)))
    for epoch_data in history:
        plt.plot(epoch_data['fpr'], epoch_data['tpr'], lw=1.5, color=colors[epoch_data['epoch']-1],
                 label=f"Epoch {epoch_data['epoch']} (AUC = {epoch_data['auc']:.4f})")
    plt.plot([0, 1], [0, 1], color='black', lw=1.5, linestyle='--', label='Chance')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('Model Progression: ROC Curves by Epoch')
    plt.legend(loc="lower right"); plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(filename, dpi=150); plt.close()

def save_publication_style_roc_curve(best_epoch_data, filename):
    plt.figure(figsize=(6, 6))
    plt.plot(best_epoch_data['fpr'], best_epoch_data['tpr'], lw=2, color='deeppink',
             label=f"Ours (AUC = {best_epoch_data['auc']:.2f})")
    plt.plot([0, 1], [0, 1], color='black', lw=1.5, linestyle='--')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.0])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC on UIT-ADrone dataset')
    plt.legend(loc="lower right"); plt.grid(False)
    plt.savefig(filename, dpi=300, bbox_inches='tight'); plt.close()

def calculate_eer(fpr, tpr):
    fnr = 1 - tpr
    eer_index = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[eer_index] + fnr[eer_index]) / 2.0
    
# ==============================================================================
# SECTION 6: TRAINING & EVALUATION LOGIC
# ==============================================================================
def train_epoch(epoch, model, data_loader, optimizer, lr_scheduler, metrics, device, config):
    model.train()
    metrics.reset()
    average_loss = []
    
    loss_l1 = nn.L1Loss().to(device)
    loss_ssim = SSIM(size_average=True).to(device)
    loss_perceptual = PerceptualLoss().to(device)

    for batch_data in tqdm(data_loader, desc=f"Training Epoch {epoch}"):
        inputs = batch_data['standard'][:, :4].to(device)
        target = batch_data['standard'][:, 4].to(device)
        optimizer.zero_grad()
        batch_pred = model(inputs)
        
        batch_pred_norm = (batch_pred + 1) / 2
        target_norm = (target + 1) / 2
        
        l1_val = loss_l1(batch_pred, target)
        ssim_val = loss_ssim(batch_pred_norm, target_norm)
        perceptual_val = loss_perceptual(batch_pred_norm, target_norm)

        loss = (config.l1_weight * l1_val +
                config.ssim_weight * (1 - ssim_val) +
                config.perceptual_weight * perceptual_val)
        
        loss.backward()
        optimizer.step()
        if lr_scheduler is not None:
            lr_scheduler.step()
        metrics.update('loss', loss.item())
        average_loss.append(loss.item())
    
    train_loss = np.mean(average_loss) if average_loss else 0
    print(f"Train Epoch: {epoch:03d} Average Loss: {train_loss:.4f}")
    return {'loss': train_loss}

def valid_epoch(epoch, model, data_loader, metrics, device, frame_save_dir, config):
    model.eval()
    metrics.reset()
    all_anomaly_scores = []
    all_labels = []
    loss_l1 = nn.L1Loss(reduction='none').to(device)
    loss_ssim = SSIM(size_average=False).to(device)

    # --- IMPORTANT: VERIFY THIS PATH ---
    label_path = '/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy'
    if not os.path.exists(label_path):
        print(f"Validation label not found at {label_path}, skipping.")
        return {'loss': -1, 'auc': 0.0, 'eer': 1.0, 'fpr': np.array([0]), 'tpr': np.array([0])}
    ground_truth_labels = np.load(label_path, allow_pickle=True)
    
    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(data_loader, desc=f"Validating Epoch {epoch}")):
            inputs = batch_data['standard'][:, :4].to(device)
            target = batch_data['standard'][:, 4].to(device)
            
            batch_pred = model(inputs)
            
            batch_pred_norm = (batch_pred + 1) / 2
            target_norm = (target + 1) / 2
            
            l1_scores = torch.mean(loss_l1(batch_pred, target).view(target.size(0), -1), dim=1)
            ssim_scores = loss_ssim(batch_pred_norm, target_norm)
            
            anomaly_scores = ((1 - config.ssim_weight) * l1_scores + 
                                config.ssim_weight * (1 - ssim_scores))
            
            all_anomaly_scores.extend(anomaly_scores.cpu().numpy())
            
            try:
                target_frame_path = data_loader.dataset.video_frames[batch_idx + 4]
                frame_number = int(os.path.basename(target_frame_path).split('.')[0].split('_')[-1])
                if frame_number < len(ground_truth_labels):
                    all_labels.append(ground_truth_labels[frame_number])
                else:
                    all_labels.append(0)
            except Exception as e:
                all_labels.append(0)

            if batch_idx < 5:
                save_anomaly_visualization(target[0:1], batch_pred[0:1], epoch, batch_idx, anomaly_scores.mean().item(), frame_save_dir)

    final_labels = all_labels[:len(all_anomaly_scores)]
    avg_loss = np.mean(all_anomaly_scores) if all_anomaly_scores else 0
    
    if len(np.unique(final_labels)) < 2:
        frame_auc, eer = 0.0, 1.0
        fpr, tpr = np.array([0, 1]), np.array([0, 1])
    else:
        frame_auc = roc_auc_score(y_true=final_labels, y_score=all_anomaly_scores)
        fpr, tpr, _ = roc_curve(y_true=final_labels, y_score=all_anomaly_scores)
        fpr = np.concatenate([[0], fpr])
        tpr = np.concatenate([[0], tpr])
        eer = calculate_eer(fpr, tpr)

    metrics.update('loss', avg_loss); metrics.update('auc', frame_auc); metrics.update('eer', eer)
    print(f"Validation Epoch: {epoch:03d}, AUC: {frame_auc:.4f}, EER: {eer:.4f}, Loss: {avg_loss:.4f}")
    return {'loss': avg_loss, 'auc': frame_auc, 'eer': eer, 'fpr': fpr, 'tpr': tpr}

def save_model(epoch, model, optimizer, lr_scheduler, device_ids, best=False, save_dir='outputs'):
    state = {'epoch': epoch,
             'state_dict': model.state_dict() if len(device_ids) <= 1 else model.module.state_dict(),
             'optimizer': optimizer.state_dict(),
             'lr_scheduler': lr_scheduler.state_dict() if lr_scheduler else None}
    checkpoints_dir = os.path.join(save_dir, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)
    if best:
        torch.save(state, os.path.join(checkpoints_dir, 'best.pth'))
    torch.save(state, os.path.join(checkpoints_dir, 'current.pth'))

# ==============================================================================
# SECTION 7: MAIN EXECUTION BLOCK
# ==============================================================================
def main():
    config = get_train_config()
    device, device_ids = setup_device(config.n_gpu)
    train_metrics = MetricTracker('loss')
    valid_metrics = MetricTracker('loss', 'auc', 'eer')

    output_dir = "outputs"
    frame_save_dir = os.path.join(output_dir, "output_frames")
    os.makedirs(frame_save_dir, exist_ok=True)
    
    log_file_path = os.path.join(output_dir, 'training_log.csv')
    with open(log_file_path, 'w') as log_file:
        log_file.write('epoch,train_loss,val_loss,val_auc,val_eer\n')
    print(f"Created log file at: {log_file_path}")

    model = VisionTransformerSOTA(image_size=config.image_size, num_frames=config.num_frames).to(device)
    if len(device_ids) > 1:
        model = nn.DataParallel(model, device_ids=device_ids)

    if bool(config.train):
        # --- IMPORTANT: VERIFY THESE PATHS ---
        train_folder = "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_train"
        test_folder = "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_test"

        data_transform = transforms.Compose([transforms.ToTensor()])
        train_dataset = DataLoader(train_folder, data_transform, config.image_size, config.image_size)
        train_batch = data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                                      num_workers=config.num_workers, drop_last=True)
        test_dataset = DataLoader(test_folder, data_transform, config.image_size, config.image_size)
        test_batch = data.DataLoader(test_dataset, batch_size=1, shuffle=False,
                                     num_workers=config.num_workers)
        
        print(f"Training data loaded: {len(train_dataset)} samples.")
        print(f"Validation data loaded: {len(test_dataset)} samples.")

        optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.wd)
        total_steps = len(train_batch) * config.epochs
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.01, end_factor=1.0, total_iters=config.warmup_steps
        )
        main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=total_steps - config.warmup_steps, eta_min=1e-6
        )
        lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[config.warmup_steps]
        )
        best_auc = 0.0
        roc_history = []
        best_epoch_data = None

        for epoch in range(1, config.epochs + 1):
            log = {'epoch': epoch}
            train_result = train_epoch(epoch, model, train_batch, optimizer, lr_scheduler, train_metrics, device, config)
            log.update(train_result)
            
            valid_result = valid_epoch(epoch, model, test_batch, valid_metrics, device, frame_save_dir, config)
            
            current_epoch_data = {'epoch': epoch, **valid_result}
            roc_history.append(current_epoch_data)
            
            log.update(**{'val_' + k: v for k, v in valid_result.items() if k not in ['fpr', 'tpr']})
            
            with open(log_file_path, 'a') as log_file:
                log_line = f"{epoch},{log['loss']:.6f},{log['val_loss']:.6f},{log['val_auc']:.6f},{log['val_eer']:.6f}\n"
                log_file.write(log_line)

            if valid_result['auc'] > best_auc:
                best_auc = valid_result['auc']
                best_epoch_data = current_epoch_data
                print(f"*** New best AUC: {best_auc:.4f} at epoch {epoch}. Saving best model. ***")
                save_model(epoch, model, optimizer, lr_scheduler, device_ids, best=True, save_dir=output_dir)
            
            save_model(epoch, model, optimizer, lr_scheduler, device_ids, best=False, save_dir=output_dir)
            
            if best_epoch_data:
                best_so_far_plot_file = os.path.join(output_dir, 'best_roc_so_far.jpg')
                title = f"Best ROC So Far (from Epoch {best_epoch_data['epoch']}, AUC={best_epoch_data['auc']:.4f})"
                save_single_roc_curve(best_epoch_data, best_so_far_plot_file, title=title)

            print("--- Epoch Summary ---")
            for key, value in log.items():
                print(f'    {str(key):15s}: {value}')
            print("---------------------")
        
        print("\n--- Training finished. ---")
        combined_plot_file = os.path.join(output_dir, 'all_epochs_roc_curve.jpg')
        save_combined_roc_curves(roc_history, combined_plot_file)
        print(f"Saved combined ROC curve plot to {combined_plot_file}")

        if best_epoch_data:
            best_epoch_plot_file = os.path.join(output_dir, 'ROC Cuve on UIT ADrone dataset.jpg')
            save_single_roc_curve(best_epoch_data, best_epoch_plot_file)
            print(f"Saved best epoch's ROC curve plot to {best_epoch_plot_file}")
            publication_plot_file = os.path.join(output_dir, 'publication_style_roc_curve.jpg')
            save_publication_style_roc_curve(best_epoch_data, publication_plot_file)
            print(f"Saved publication-style ROC curve plot to {publication_plot_file}")

if __name__ == '__main__':
    main()

----------------- Config ---------------
----------------- End -------------------
Created log file at: outputs/training_log.csv
Training data loaded: 881 samples.
Validation data loaded: 2349 samples.


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Training Epoch 1:   0%|          | 0/110 [00:00<?, ?it/s]/tmp/ipykernel_96/1371599158.py:143: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the ori

Train Epoch: 001 Average Loss: 0.8620


Validating Epoch 1: 100%|██████████| 2349/2349 [04:14<00:00,  9.23it/s]
/tmp/ipykernel_96/1371599158.py:143: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self._data.total[key] += value * n
/tmp/ipykernel_96/1371599158.py:144: FutureWarning:

Validation Epoch: 001, AUC: 0.3913, EER: 0.5955, Loss: 0.5514
*** New best AUC: 0.3913 at epoch 1. Saving best model. ***
--- Epoch Summary ---
    epoch          : 1
    loss           : 0.8619802139022134
    val_loss       : 0.5513604283332825
    val_auc        : 0.39132629029002086
    val_eer        : 0.5954692051102117
---------------------


...

Validating Epoch 7: 100%|██████████| 2349/2349 [03:50<00:00, 10.19it/s]
/tmp/ipykernel_96/1371599158.py:143: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self._data.total[key] += value * n
/tmp/ipykernel_96/1371599158.py:144: FutureWarning:

Validation Epoch: 007, AUC: 0.7206, EER: 0.3256, Loss: 0.3620
--- Epoch Summary ---
    epoch          : 7
    loss           : 0.48172041502865875
    val_loss       : 0.3620152771472931
    val_auc        : 0.7205869629999904
    val_eer        : 0.325550097570823
---------------------


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Training Epoch 8:   0%|          | 0/110 [00:00<?, ?it/s]/tmp/ipykernel_96/1371599158.py:143: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the ori

Train Epoch: 008 Average Loss: 0.4675


Validating Epoch 8:  62%|██████▏   | 1467/2349 [02:27<01:31,  9.64it/s]